# Bro-modellen på Trondheim-data

Laster bro-checkpoint og kjører modellen på en Trondheim-tile med bro.
Følger samme mønster som `bro_visualization.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import hydra
import laspy
import numpy as np
import torch

PROJECT_ROOT = Path('/cluster/home/larshfle/superpoint_transformer_new')
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.utils import init_config
from src.transforms import *
from src.data import *

print("Imports successful!")

## 1. Paths

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

TILE_LAZ  = Path('/cluster/home/larshfle/datasets/trondheim2022_30pkt/raw/train/32-1-511-215-35.laz')
CKPT_PATH = str(PROJECT_ROOT / 'logs/train/runs/2026-05-06_19-01-24/checkpoints/last.ckpt')

# Midlertidig bro-katalog der vi legger Trondheim-tilen
TMP_DATA_DIR = Path('/tmp/bro_trondheim')
TILE_ID      = 'trondheim_32-1-511-215-35'

assert TILE_LAZ.exists(),  f'Finner ikke: {TILE_LAZ}'
assert Path(CKPT_PATH).exists(), f'Finner ikke: {CKPT_PATH}'
print(f'Device: {device}')
print(f'Tile: {TILE_LAZ.name}')
print(f'Checkpoint: {CKPT_PATH}')

## 2. Skriv LAZ som LAS til midlertidig bro-katalog

In [ ]:
las_out = TMP_DATA_DIR / 'raw' / 'test' / f'{TILE_ID}.las'
las_out.parent.mkdir(parents=True, exist_ok=True)
(TMP_DATA_DIR / 'raw' / 'train').mkdir(parents=True, exist_ok=True)
(TMP_DATA_DIR / 'raw' / 'val').mkdir(parents=True, exist_ok=True)

if not las_out.exists():
    print('Konverterer LAZ → LAS...')
    las = laspy.read(str(TILE_LAZ))
    las.write(str(las_out))
    print(f'Lagret: {las_out}')
else:
    print(f'LAS finnes allerede: {las_out}')

las = laspy.read(str(las_out))
cls = np.asarray(las.classification)
n_bridge = (cls == 17).sum()
print(f'Punkter: {len(cls):,}  |  Bropunkter (LAS 17): {n_bridge:,} ({100*n_bridge/len(cls):.3f}%)')

## 3. Last config og datamodule — patch TILES til å inkludere Trondheim-tilen

In [ ]:
import src.datasets.bro_config as bro_config
import src.datasets.bro as bro_module

new_tiles = {'train': [], 'val': [], 'test': [TILE_ID]}
bro_config.TILES = new_tiles
bro_module.TILES = new_tiles

cfg = init_config(overrides=[
    'experiment=semantic/bro',
    f'ckpt_path={CKPT_PATH}',
    'datamodule.mini=false',
    'datamodule.load_full_res_idx=true',
    'datamodule.prepare_only_test=true',
    'datamodule.xy_tiling=3',
])
cfg.datamodule.data_dir = str(TMP_DATA_DIR)

datamodule = hydra.utils.instantiate(cfg.datamodule)
datamodule.prepare_data()
datamodule.setup()

dataset = datamodule.test_dataset
print(f'Dataset lastet: {len(dataset)} tile(r)')
dataset.print_classes()

## 4. Last modell

In [ ]:
model = hydra.utils.instantiate(cfg.model)

load_kwargs = {}
pretrained_cnn_ckpt_path = cfg.datamodule.get('pretrained_cnn_ckpt_path', None)
if pretrained_cnn_ckpt_path is not None:
    load_kwargs['pretrained_cnn_ckpt_path'] = pretrained_cnn_ckpt_path

model = model._load_from_checkpoint(cfg.ckpt_path, **load_kwargs)
model = model.eval().to(device)

print('Model loaded from checkpoint!')
print(f'Device: {device}')

## 5. Kjør inferens — identisk med bro_visualization.ipynb

In [ ]:
model.net.store_features = True

subtile_ids = [i for i, cid in enumerate(dataset.cloud_ids) if cid.startswith(TILE_ID)]
print(f'Antall subtiler: {len(subtile_ids)}')
for i, idx in enumerate(subtile_ids):
    print(f'  {i}: {dataset.cloud_ids[idx]}')

nags, outputs = [], []
for idx in subtile_ids:
    n = dataset[idx]
    n = dataset.on_device_transform(n.to(device))
    with torch.no_grad():
        out = model(n)
    n[0].semantic_pred = out.voxel_semantic_pred(super_index=n[0].super_index)
    nags.append(n)
    outputs.append(out)
print(f'Inferens ferdig på {len(nags)} subtiler!')


In [ ]:
# Aggregerte stats over alle subtiler
tp_tot, fp_tot, fn_tot = 0, 0, 0
for n in nags:
    gt   = n[0].y.argmax(dim=-1).cpu().numpy()
    pred = n[0].semantic_pred.cpu().numpy()
    valid = gt < BRO_NUM_CLASSES
    tp_tot += int(((gt[valid]==1) & (pred[valid]==1)).sum())
    fp_tot += int(((gt[valid]==0) & (pred[valid]==1)).sum())
    fn_tot += int(((gt[valid]==1) & (pred[valid]==0)).sum())

iou  = tp_tot/(tp_tot+fp_tot+fn_tot) if (tp_tot+fp_tot+fn_tot)>0 else float('nan')
prec = tp_tot/(tp_tot+fp_tot)         if (tp_tot+fp_tot)>0         else float('nan')
rec  = tp_tot/(tp_tot+fn_tot)         if (tp_tot+fn_tot)>0         else float('nan')
print(f'Bridge IoU:       {100*iou:.2f}%')
print(f'Bridge Precision: {100*prec:.2f}%')
print(f'Bridge Recall:    {100*rec:.2f}%')
print(f'GT bridge pts:    {tp_tot+fn_tot:,}')


## 6. Visualisering — ground truth

In [ ]:
i = 0  # endre 0-N for å velge subtile
nags[i].show(
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=200000,
)


In [ ]:
# Gammel visualisering (kommentert ut — bruk celle 14 og 19 med nags[i])
# nag.show(
#     class_names=dataset.class_names,
#     class_colors=dataset.class_colors,
#     center=[0, 0, 0],
#     radius=50,
#     max_points=200000,
# )


In [ ]:
i = 0  # subtile å eksportere
output_path = f'/cluster/home/larshfle/superpoint_transformer_new/bro_visualization_on_trondheim_{i}.html'

nags[i].show(
    figsize=1600,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=200000,
    semantic_pred=True,
    title=f'Bro på Trondheim — subtile {i}',
    path=output_path,
)
print(f'Eksportert: {output_path}')


In [ ]:
# import numpy as np
# 
# gt   = nag[0].y.argmax(dim=-1).cpu().numpy()
# pred = nag[0].semantic_pred.cpu().numpy()
# 
# valid = gt < 2
# 
# tp = int(((gt[valid] == 1) & (pred[valid] == 1)).sum())
# fp = int(((gt[valid] == 0) & (pred[valid] == 1)).sum())
# fn = int(((gt[valid] == 1) & (pred[valid] == 0)).sum())
# 
# iou       = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float('nan')
# precision = tp / (tp + fp)       if (tp + fp) > 0       else float('nan')
# recall    = tp / (tp + fn)       if (tp + fn) > 0       else float('nan')
# 
# print(f'Bridge IoU:       {100*iou:.2f}%')
# print(f'Bridge Precision: {100*precision:.2f}%')
# print(f'Bridge Recall:    {100*recall:.2f}%')
# print(f'GT bridge pts:    {tp+fn:,}')


## 7. Visualisering — prediksjoner

In [ ]:
i = 0  # endre 0-N for å velge subtile
nags[i].show(
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=200000,
    semantic_pred=True,
)
